# 02 数据清洗（列名统一 & 类型修复）
本 Notebook 完成：
1. 单表清洗
2. 宽表与长表转换
3. 多表合并
4. 清洗后数据存为 CSV 和 Parquet（列名英文，类型正确）
5. Parquet 特性演示与性能对比

**修复**：
- 清洗后列名统一为英文（date, code, open, close, …）
- Parquet 写入时使用显式 pyarrow schema，避免 `large_string` 及列名不匹配

In [ ]:
import pandas as pd
import numpy as np
import os
import time
import pyarrow as pa
import pyarrow.parquet as pq

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

## 1. 读取原始数据

In [ ]:
DATA_DIR = "dshw-p01/data"

stock_files = [f for f in os.listdir(f"{DATA_DIR}/stock") if f.endswith('.csv')]
stock_dfs = {}
for f in stock_files:
    code = f.replace('stock_', '').replace('.csv', '')
    df = pd.read_csv(f"{DATA_DIR}/stock/{f}")
    stock_dfs[code] = df
    print(f"已读取 {code}: shape={df.shape}")

index_files = [f for f in os.listdir(f"{DATA_DIR}/index") if f.endswith('.csv')]
index_dfs = {}
for f in index_files:
    idx_name = f.replace('index_', '').replace('.csv', '')
    df = pd.read_csv(f"{DATA_DIR}/index/{f}")
    index_dfs[idx_name] = df
    print(f"已读取指数 {idx_name}: shape={df.shape}")

macro_cpi = pd.read_csv(f"{DATA_DIR}/macro/macro_cpi.csv")
macro_m2 = pd.read_csv(f"{DATA_DIR}/macro/macro_m2.csv")
print(f"CPI 数据: {macro_cpi.shape}")
print(f"M2 数据:  {macro_m2.shape}")

## 2. 单表清洗

### 2.1 缺失值检测与处理

In [ ]:
demo_code = '601398'
df_demo = stock_dfs[demo_code].copy()
missing = df_demo.isnull().sum()
missing_ratio = (missing / len(df_demo)) * 100
missing_table = pd.DataFrame({'缺失数量': missing, '缺失比例(%)': missing_ratio})
print(f"{demo_code} 缺失值统计：")
display(missing_table)

# 缺失值处理：向前填充
for code, df in stock_dfs.items():
    df.ffill(inplace=True)
    df.dropna(inplace=True)
    stock_dfs[code] = df
print("缺失值处理完成。")

### 2.2 日期格式统一与设为索引

In [ ]:
for code, df in stock_dfs.items():
    df['日期'] = pd.to_datetime(df['日期'])
    df.set_index('日期', inplace=True)
    df.sort_index(inplace=True)
    stock_dfs[code] = df
print("日期已统一为 datetime64 并设为索引。")
display(stock_dfs['601398'].head())

### 2.3 数据类型检查

In [ ]:
for code, df in stock_dfs.items():
    for col in ['开盘', '收盘', '最高', '最低', '成交量', '成交额']:
        if df[col].dtype == 'object':
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df.ffill(inplace=True)
    df.dropna(inplace=True)
    stock_dfs[code] = df
print("数据类型检查完毕。")

### 2.4 重复值处理

In [ ]:
dup_total = 0
for code, df in stock_dfs.items():
    dup = df.index.duplicated().sum()
    if dup > 0:
        dup_total += dup
        df = df[~df.index.duplicated(keep='first')]
        stock_dfs[code] = df
        print(f"{code} 发现重复行 {dup} 条，已删除。")
if dup_total == 0:
    print("未发现重复行。")

### 2.5 离群值标注

In [ ]:
for code, df in stock_dfs.items():
    df['return'] = df['收盘'].pct_change()
    df['is_extreme'] = df['return'].abs() > 0.20
    extreme_cnt = df['is_extreme'].sum()
    if extreme_cnt > 0:
        print(f"{code} 出现 {extreme_cnt} 个极端涨跌幅（>±20%），已标注。")
print("离群值标注完成。")

## 3. 宽表与长表转换

In [ ]:
close_series = {code: df['收盘'].rename(code) for code, df in stock_dfs.items()}
wide_close = pd.concat(close_series, axis=1)
wide_close.index.name = 'date'
print("宽表（收盘价）形状：", wide_close.shape)
display(wide_close.head())

long_close = wide_close.reset_index().melt(id_vars='date', var_name='code', value_name='close')
long_close['date'] = pd.to_datetime(long_close['date'])
print("长表形状：", long_close.shape)
display(long_close.head())

## 4. 多表合并

In [ ]:
index_close = {}
for idx_code, df_idx in index_dfs.items():
    df_idx['date'] = pd.to_datetime(df_idx['date'])
    df_idx.set_index('date', inplace=True)
    index_close[idx_code] = df_idx['close'].rename(idx_code)
index_wide = pd.concat(index_close.values(), axis=1)
index_wide.index.name = 'date'

combined_stock_index = wide_close.join(index_wide, how='left')
print(f"left join 后行数: {len(combined_stock_index)}")
display(combined_stock_index.head())

In [ ]:
macro_cpi['date'] = pd.to_datetime(macro_cpi['date'], format='%Y-%m')
macro_m2['date'] = pd.to_datetime(macro_m2['date'], format='%Y-%m')
macro_cpi['month_start'] = macro_cpi['date'].dt.to_period('M').dt.start_time
macro_m2['month_start'] = macro_m2['date'].dt.to_period('M').dt.start_time

if not isinstance(combined_stock_index.index, pd.DatetimeIndex):
    combined_stock_index.index = pd.to_datetime(combined_stock_index.index)
combined_stock_index['month_start'] = combined_stock_index.index.to_period('M').to_timestamp()

combined = combined_stock_index.merge(macro_cpi[['month_start', 'cpi_yoy']], on='month_start', how='left')
combined = combined.merge(macro_m2[['month_start', 'm2_yoy']], on='month_start', how='left')
combined.drop(columns='month_start', inplace=True)
print(f"最终合并数据行数: {len(combined)}")
display(combined.head())

## 5. 保存清洗后数据（CSV + Parquet，英文列名 + 显式 schema）

In [ ]:
CLEAN_DIR = os.path.join(DATA_DIR, 'clean')
os.makedirs(CLEAN_DIR, exist_ok=True)

# 构建清洗后的长表，并统一列名为英文
col_map = {'开盘': 'open', '收盘': 'close', '最高': 'high', '最低': 'low',
           '成交量': 'volume', '成交额': 'amount'}

stock_long_list = []
for code, df in stock_dfs.items():
    df_temp = df.reset_index().rename(columns={'日期': 'date'})
    df_temp.rename(columns=col_map, inplace=True)
    df_temp['code'] = code
    # 只保留需要的列
    keep_cols = ['date', 'code', 'open', 'close', 'high', 'low', 'volume', 'amount', 'return', 'is_extreme']
    df_temp = df_temp[keep_cols]
    stock_long_list.append(df_temp)

stock_clean = pd.concat(stock_long_list, ignore_index=True)
stock_clean['date'] = pd.to_datetime(stock_clean['date'])
stock_clean['code'] = stock_clean['code'].astype('string')

# 保存 CSV
csv_path = os.path.join(CLEAN_DIR, 'stock_clean.csv')
stock_clean.to_csv(csv_path, index=False)

# 保存 Parquet —— 使用显式 pyarrow schema，确保类型正确
parquet_path = os.path.join(CLEAN_DIR, 'stock_clean.parquet')

# 定义 schema
pa_schema = pa.schema([
    ('date', pa.timestamp('ns')),
    ('code', pa.string()),
    ('open', pa.float64()),
    ('close', pa.float64()),
    ('high', pa.float64()),
    ('low', pa.float64()),
    ('volume', pa.float64()),
    ('amount', pa.float64()),
    ('return', pa.float64()),
    ('is_extreme', pa.bool_()),
])

# 将 DataFrame 转为 pyarrow Table（只取 schema 中存在的列）
table = pa.Table.from_pandas(stock_clean[list(pa_schema.names)], schema=pa_schema)
pq.write_table(table, parquet_path)

# 保存合并数据
combined_path_csv = os.path.join(DATA_DIR, 'combined', 'combined_data.csv')
combined.to_csv(combined_path_csv)
combined_path_parquet = os.path.join(DATA_DIR, 'combined', 'combined_data.parquet')
combined.to_parquet(combined_path_parquet)

print("全部数据已保存。")
print(f"CSV: {csv_path}")
print(f"Parquet: {parquet_path}")

## 6. Parquet 特性演示与性能对比

In [ ]:
# 检查文件存在
if not os.path.exists(parquet_path):
    raise FileNotFoundError(f"Parquet 文件未找到: {parquet_path}")

# 列式读取（现在列名均为英文，不存在 close 缺失问题）
df_part = pd.read_parquet(parquet_path, columns=['date', 'code', 'close'])
print("列式读取（仅 date, code, close）：")
display(df_part.head())

In [ ]:
# 查看 Schema
schema_read = pq.read_schema(parquet_path)
print("Parquet Schema：")
print(schema_read)

In [ ]:
# 性能对比
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"CSV 文件未找到: {csv_path}")

csv_size = os.path.getsize(csv_path) / 1024
parquet_size = os.path.getsize(parquet_path) / 1024

t0 = time.time()
pd.read_csv(csv_path)
csv_time = time.time() - t0

t0 = time.time()
pd.read_parquet(parquet_path)
parquet_time = time.time() - t0

print(f"CSV    读取耗时: {csv_time:.3f}s  |  文件大小: {csv_size:.1f} KB")
print(f"Parquet 读取耗时: {parquet_time:.3f}s  |  文件大小: {parquet_size:.1f} KB")
print(f"速度提升: {csv_time/parquet_time:.2f}x  体积缩减: {csv_size/parquet_size:.2f}x")

**分析**：数据量小时差异不明显；当数据量巨大、只读取部分列时，Parquet 优势显著。

In [ ]:
print("清洗完成！")